In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
from datasets import load_dataset_builder
# dataset_name = "sms_spam"
dataset_name = "SetFit/enron_spam"
load_dataset_builder(dataset_name).info

d:\fine-tune-text-classification\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\datasets--SetFit--enron_spam. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Repo card metadata block was not found. Setting CardData to empty.


DatasetInfo(description='', citation='', homepage='', license='', features={'message_id': Value('int64'), 'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string'), 'subject': Value('string'), 'message': Value('string'), 'date': Value('timestamp[s]')}, post_processed=None, supervised_keys=None, builder_name='json', dataset_name='enron_spam', config_name='default', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=96980930, num_examples=31716, shard_lengths=None, original_shard_lengths=None, dataset_name='enron_spam'), 'test': SplitInfo(name='test', num_bytes=6013617, num_examples=2000, shard_lengths=None, original_shard_lengths=None, dataset_name='enron_spam')}, download_checksums={'hf://datasets/SetFit/enron_spam@1916f66c89d52221ae33eb57d44498b4f3a5df22/train.jsonl': {'num_bytes': 101069043, 'checksum': None}, 'hf://datasets/SetFit/enron_spam@1916f66c89d52221ae33eb57d44498b4f3a5df22/test.jsonl': {'num_bytes': 6273613, 'checksum': None}}, download_

In [16]:
from datasets import load_dataset

dataset = load_dataset(dataset_name)
# 5,574 SMS messages (747 spam, 4,827 ham)

Repo card metadata block was not found. Setting CardData to empty.
Generating test split: 100%|██████████| 2000/2000 [00:00<00:00, 219786.94 examples/s]


In [18]:
dataset = dataset['train'].train_test_split(test_size=0.2)  # Split train/test

In [19]:
spam_examples = dataset['test'].filter(lambda x: x['label'] == 1)
spam_examples[0]

Filter: 100%|██████████| 5075/5075 [00:00<00:00, 58891.16 examples/s]


{'message_id': 5058,
 'text': 'mystery shopping - extra casual income customer intelligence strategies\nfrontline focus international inc .\nlink house , 140 the broadway\nsuite # 12 surbiton , surrey\nkt 6 7 je\n64 king st east hamilton ontario canada\nhamilton , ontario , canada\nl 8 n la 6\ncompany\nservices :\nconsultants in consumer care\nposition : mystery shopping\nsecret shopper\njob\ndescription :\nmystery shoppers provide objective factual feedback about the quality of\ntheir experiences at local retail stores , restaurants , theatres and hotels or\nother establishments that deal with the public . ( all within your city or local area )\nvisit\nlocal establishments to provide feedback on customer service and / or quality . online reporting of\nresults within 24 hrs .\nsalary : vary per project .\nqualifications : access to computer with internet access\nand email address . effective communication\nskills . must enjoy working with\nthe public . no experience necessary .\nonline

# Modified Logistic Regression

## TFIDF solution

Note: 
- this gonna be tricky if we have mixed Thai and English, see 07_tokenization.ipynb for solution
- more training not help because less training performs better

In [7]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from package.models.modified_logistic_regression import ModifiedLogisticRegressionPU

# 1. Load data
dataset = load_dataset("SetFit/enron_spam")
train_data = dataset['train']

# 2. Create PU scenario (hide 75% of spam)
spam_indices = [i for i, label in enumerate(train_data['label']) if label == 1]
ham_indices = [i for i, label in enumerate(train_data['label']) if label == 0]

# Keep only 25% of spam labeled
np.random.seed(42)
labeled_spam = np.random.choice(spam_indices, size=len(spam_indices)//4, replace=False)

# Create PU labels
pu_labels = np.zeros(len(train_data))
pu_labels[labeled_spam] = 1  # Only 25% spam labeled as 1, rest are 0 (unlabeled)

# 3. Vectorize text
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(train_data['text']).toarray()
y_true = np.array(train_data['label'])  # True labels (for evaluation only)
y_pu = pu_labels  # PU labels (for training)

# 4. Train Modified Logistic Regression
model = ModifiedLogisticRegressionPU(
    lr=0.001,
    epochs=10000,
    verbose=True
)

model.fit(X, y_pu)

# 5. Predict
y_pred = model.predict(X)
y_proba = model.predict_proba(X)[:, 1]

# 6. Evaluate
print(f"\nEstimated c (labeling rate): {model.c_hat_:.4f}")
print(f"True c: {len(labeled_spam) / len(spam_indices):.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Ham', 'Spam']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# 7. Check probability distribution
import pandas as pd
results = pd.DataFrame({
    'true_label': y_true,
    'pu_label': y_pu,
    'predicted_proba': y_proba
})

print("\nProbability by True Label:")
print(results.groupby('true_label')['predicted_proba'].describe())


Repo card metadata block was not found. Setting CardData to empty.


Epoch 0  Loss 0.6931
Epoch 100  Loss 0.5281
Epoch 200  Loss 0.4527
Epoch 300  Loss 0.4165
Epoch 400  Loss 0.3964
Epoch 500  Loss 0.3830
Epoch 600  Loss 0.3729
Epoch 700  Loss 0.3646
Epoch 800  Loss 0.3576
Epoch 900  Loss 0.3515
Epoch 1000  Loss 0.3463
Epoch 1100  Loss 0.3416
Epoch 1200  Loss 0.3376
Epoch 1300  Loss 0.3340
Epoch 1400  Loss 0.3308
Epoch 1500  Loss 0.3279
Epoch 1600  Loss 0.3253
Epoch 1700  Loss 0.3230
Epoch 1800  Loss 0.3208
Epoch 1900  Loss 0.3189
Epoch 2000  Loss 0.3171
Epoch 2100  Loss 0.3155
Epoch 2200  Loss 0.3139
Epoch 2300  Loss 0.3125
Epoch 2400  Loss 0.3112
Epoch 2500  Loss 0.3100
Epoch 2600  Loss 0.3089
Epoch 2700  Loss 0.3078
Epoch 2800  Loss 0.3068
Epoch 2900  Loss 0.3059
Epoch 3000  Loss 0.3050
Epoch 3100  Loss 0.3041
Epoch 3200  Loss 0.3033
Epoch 3300  Loss 0.3025
Epoch 3400  Loss 0.3018
Epoch 3500  Loss 0.3011
Epoch 3600  Loss 0.3005
Epoch 3700  Loss 0.2998
Epoch 3800  Loss 0.2992
Epoch 3900  Loss 0.2986
Epoch 4000  Loss 0.2980
Epoch 4100  Loss 0.2975
Epoc

## Embeddings with PCA solution

Note: 
- it looks promising only in more training
- it still needs to find the feasible way to inspect the answers

In [8]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from package.models.modified_logistic_regression import ModifiedLogisticRegressionPU
from sklearn.decomposition import PCA

# 1. Load BGE model
print("Loading BGE model...")
embedder = SentenceTransformer('BAAI/bge-small-en-v1.5')  # Or bge-base/large

# 2. Load data
dataset = load_dataset("SetFit/enron_spam")
train_data = dataset['train']

# 3. Create PU scenario (hide 75% of spam)
spam_indices = [i for i, label in enumerate(train_data['label']) if label == 1]
ham_indices = [i for i, label in enumerate(train_data['label']) if label == 0]

np.random.seed(42)
labeled_spam = np.random.choice(spam_indices, size=len(spam_indices)//4, replace=False)

pu_labels = np.zeros(len(train_data))
pu_labels[labeled_spam] = 1

# 4. Generate embeddings (this takes time!)
print("Generating embeddings...")
texts = train_data['text']
_X = embedder.encode(texts, show_progress_bar=True, batch_size=32)

# Reduce dimensions to match TF-IDF sparsity
pca = PCA(n_components=100)  # 384 → 100 dims
X = pca.fit_transform(_X)

y_true = np.array(train_data['label'])
y_pu = pu_labels

# 5. Train Modified Logistic Regression
print("Training PU model...")
model = ModifiedLogisticRegressionPU(
    lr=0.001,
    epochs=10000,
    verbose=True
)

model.fit(X, y_pu)

# 6. Predict
y_pred = model.predict(X)
y_proba = model.predict_proba(X)[:, 1]

# 7. Evaluate
print(f"\nEstimated c: {model.c_hat_:.4f}")
print(f"True c: {len(labeled_spam) / len(spam_indices):.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Ham', 'Spam']))

# 8. Probability analysis
import pandas as pd
results = pd.DataFrame({
    'true_label': y_true,
    'pu_label': y_pu,
    'predicted_proba': y_proba
})

print("\nProbability Distribution:")
print(results.groupby('true_label')['predicted_proba'].describe())

Loading BGE model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11704.27it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Repo card metadata block was not found. Setting CardData to empty.


Generating embeddings...


Batches: 100%|██████████| 992/992 [00:42<00:00, 23.07it/s] 


Training PU model...
Epoch 0  Loss 0.6903
Epoch 100  Loss 0.6745
Epoch 200  Loss 0.6483
Epoch 300  Loss 0.6152
Epoch 400  Loss 0.5808
Epoch 500  Loss 0.5488
Epoch 600  Loss 0.5207
Epoch 700  Loss 0.4969
Epoch 800  Loss 0.4769
Epoch 900  Loss 0.4601
Epoch 1000  Loss 0.4461
Epoch 1100  Loss 0.4344
Epoch 1200  Loss 0.4245
Epoch 1300  Loss 0.4161
Epoch 1400  Loss 0.4091
Epoch 1500  Loss 0.4031
Epoch 1600  Loss 0.3979
Epoch 1700  Loss 0.3935
Epoch 1800  Loss 0.3898
Epoch 1900  Loss 0.3865
Epoch 2000  Loss 0.3836
Epoch 2100  Loss 0.3811
Epoch 2200  Loss 0.3790
Epoch 2300  Loss 0.3770
Epoch 2400  Loss 0.3754
Epoch 2500  Loss 0.3739
Epoch 2600  Loss 0.3725
Epoch 2700  Loss 0.3713
Epoch 2800  Loss 0.3703
Epoch 2900  Loss 0.3693
Epoch 3000  Loss 0.3684
Epoch 3100  Loss 0.3676
Epoch 3200  Loss 0.3668
Epoch 3300  Loss 0.3661
Epoch 3400  Loss 0.3654
Epoch 3500  Loss 0.3648
Epoch 3600  Loss 0.3642
Epoch 3700  Loss 0.3636
Epoch 3800  Loss 0.3631
Epoch 3900  Loss 0.3626
Epoch 4000  Loss 0.3620
Epoch 4